In [1]:
import pandas as pd
from ast import literal_eval
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
import re
import json
import glob
from ast import literal_eval
from statistics import mean
plt.rcParams["font.sans-serif"]=["SimHei"] #设置字体
plt.rcParams["axes.unicode_minus"]=False #该语句解决图像中的“-”负号的乱码问题
import matplotlib
print(matplotlib.matplotlib_fname())
print(matplotlib.get_cachedir())

/root/anaconda3/envs/llumnix/lib/python3.10/site-packages/matplotlib/mpl-data/matplotlibrc
/root/.cache/matplotlib


## 分组
将instance.csv中的profiling_data分为开为4列，同时安装instance_id进行分组

In [11]:
def get_profiling_data(filename,):
    instance_log = pd.read_csv(filename)
    # 删除dispatch_load_metric为-inf的行
    instance_log = instance_log[instance_log['dispatch_load_metric'] != -np.inf]

    # 将profiling_data列(inference_type,num_seqs,running_seq_lens,last_inference_latency)中的内容转化为4列
    instance_log[['profiling_inference_type', 'profiling_num_seqs', 'running_seq_lens', 'last_inference_latency']] = (
        instance_log['profiling_data']
        .apply(lambda x: literal_eval(x) if pd.notnull(x) else ("", None, None, None))
        .apply(pd.Series)
    )
    # 根据seq_lens列中的字符串转换为列表，得到最大序列长度
    instance_log['max_seq_len'] = instance_log['seq_lens'].apply(lambda x: max(literal_eval(x)) if pd.notnull(x) and len(literal_eval(x))>0 else 0)
    instance_log['avg_seq_len'] = instance_log['seq_lens'].apply(lambda x: mean(literal_eval(x)) if pd.notnull(x) and len(literal_eval(x))>0 else 0)

    instance_log_group = instance_log.groupby("instance_id")
    # 一个group保存为一个sheet
    with pd.ExcelWriter(filename.replace('.csv', '_metrics.xlsx')) as writer:
        for i, (instance_id, group) in enumerate(instance_log_group):
            group.to_excel(writer, sheet_name=f'Instance_{instance_id}', index=False)
            # max_seq_len非0个数
            count_max_seq_len = group['max_seq_len'].ne(0).sum()
            # 对last_inference_latency去重
            unique_latency = group['last_inference_latency'].drop_duplicates().reset_index(drop=True)
            # 个数
            count_latency = unique_latency.count()
            # 计算平均值
            avg_latency = unique_latency.mean()
            # 计算最大值
            max_latency = unique_latency.max()
            # 计算最小值
            min_latency = unique_latency.min()
            # 计算标准差
            std_latency = unique_latency.std()
            
            print(f"Instance {instance_id} - count_max_seq_len: {count_max_seq_len}, Unique Latency Count: {count_latency}, Avg: {avg_latency}, Max: {max_latency}, Min: {min_latency}, Std: {std_latency}")

# get_profiling_data('/workspace/llm-serve/Llumnix/benchmark_test/logs/A6000-t2-pdd-4/llama-7b/poisson/serve_4_tp1_1000_qps_6_instance.csv')
get_profiling_data('/workspace/llm-serve/Llumnix/benchmark_test/logs/A6000-2-formal2-concurrency-1-pdd-4/llama-7b/poisson/serve_pdd_tp1_2000_qps_4_1_3_instance.csv')
get_profiling_data('/workspace/llm-serve/Llumnix/benchmark_test/logs/A6000-2-formal2-concurrency-1-pdd-4/llama-7b/poisson/serve_pdd_tp1_2000_qps_4_2_2_instance.csv')
get_profiling_data('/workspace/llm-serve/Llumnix/benchmark_test/logs/A6000-2-formal2-concurrency-1-pdd-4/llama-7b/poisson/serve_pdd_tp1_2000_qps_4_3_1_instance.csv')

/root/anaconda3/envs/llumnix/lib/python3.10/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


Instance d0cd8e4db9794acf9527dc9c450f1d2c - count_max_seq_len: 9462, Unique Latency Count: 8783, Avg: 42.67945032272851, Max: 73.19068908691406, Min: 0.0, Std: 6.070408483367482
Instance d3efa9e06dd04155837c67cac058a72b - count_max_seq_len: 9354, Unique Latency Count: 8717, Avg: 42.692326481932824, Max: 87.78047561645508, Min: 0.0, Std: 5.857289031016772
Instance d66efcbe1e5945ab9d419dbb37fac5d3 - count_max_seq_len: 2149, Unique Latency Count: 1852, Avg: 60.06672701619356, Max: 238.34800720214844, Min: 0.0, Std: 28.747269101116416
Instance faac4afd191c4c0fa9dfe78734049908 - count_max_seq_len: 9389, Unique Latency Count: 8682, Avg: 43.15411503538507, Max: 75.41203498840332, Min: 0.0, Std: 5.618792956511199


/root/anaconda3/envs/llumnix/lib/python3.10/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


Instance 02acf160f32d4000868d7240c8a0f300 - count_max_seq_len: 1217, Unique Latency Count: 1079, Avg: 58.49944318854444, Max: 230.25059700012207, Min: 0.0, Std: 28.678641340058572
Instance 3d53a276666149639a9d9fc13e3a4879 - count_max_seq_len: 8941, Unique Latency Count: 6912, Avg: 63.28303936041064, Max: 468.78695487976074, Min: 0.0, Std: 17.257409876391137
Instance 6aa8fb96795d40b2a93465433a279c42 - count_max_seq_len: 878, Unique Latency Count: 798, Avg: 59.08269093448954, Max: 236.01555824279785, Min: 0.0, Std: 28.24686444624493
Instance f8a4485fa1b84901908ff5516cd4b351 - count_max_seq_len: 9091, Unique Latency Count: 7043, Avg: 63.219730173273724, Max: 347.37396240234375, Min: 0.0, Std: 16.110652135951877


/root/anaconda3/envs/llumnix/lib/python3.10/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


Instance 06881796b5cd4a65902d904caf87d87c - count_max_seq_len: 350, Unique Latency Count: 334, Avg: 55.92713598719614, Max: 211.55047416687012, Min: 0.0, Std: 27.955737392811226
Instance 64389f11a7ea41b0bfb1560a5b666d6b - count_max_seq_len: 939, Unique Latency Count: 861, Avg: 56.91048686374217, Max: 236.0248565673828, Min: 0.0, Std: 27.37650668411839
Instance b60ee45f5dee4f02b945908881a429e7 - count_max_seq_len: 742, Unique Latency Count: 686, Avg: 57.21615250534636, Max: 236.04965209960938, Min: 0.0, Std: 30.770558757502933
Instance d6f7a863642140ac8c2032fd630474ef - count_max_seq_len: 15893, Unique Latency Count: 9744, Avg: 79.3988973098044, Max: 399.2016315460205, Min: 0.0, Std: 24.601726397624326


## prefill/第n个token分析
从per_token_latency_breakdown_list提取中提取每个请求的第n个token的各步骤时延

In [3]:

num_req = 2000
qps=12
json_file = f'/workspace/llm-serve/Llumnix/benchmark_test/logs/A6000-t2-multi-port-zmp-pdd-4/llama-7b/poisson/benchmark_4_tp1_{num_req}_qps_{qps}_latency_info.json'
json_file = '/workspace/llm-serve/Llumnix/benchmark_test/logs/l40-pdd-hetero/llama-7b/poisson/benchmark_pdd_2000_qps_6_1,1_2_latency_info.json'
json_file = '/workspace/llm-serve/Llumnix/benchmark_test/logs/L40-test-concurrency-1-128-256/llama-7b/poisson/benchmark_pdd_2000_qps_2_1_1_latency_info.json'
with open(json_file, 'r') as f:
    data = json.load(f)
assert len(data) == 1, "Expected data to contain only one entry"
per_token_latency_breakdown_list = data[0]['per_token_latency_breakdown_list']
# df = pd.DataFrame(per_token_latency_breakdown_list[823])
# df.to_csv(json_file.replace('.json', f'_req_{823}_all.csv'), index=True)
# rows保存每个请求的prefill相关的数据
rows = []
decode_no = 0
for req_idx in range(num_req):
    # 将per_token_latency_breakdown_list[req_idx]的第一行(prefill相关)加入
    tmp = min(len(per_token_latency_breakdown_list[req_idx])-1, decode_no)
    rows.append(per_token_latency_breakdown_list[req_idx][tmp])
df = pd.DataFrame(rows, index=data[0]['request_ids'])
df.to_csv(json_file.replace('.json', f'_decode_{decode_no}.csv'), index=True)
# df.head(n=128)
    # df.to_csv(os.path.join('/workspace/llm-serve/Llumnix/logs/l40-pdd--2/llama-2-7b/uniform/' + migrate_backend, f'request_{req_idx}_timestamps.csv'), index=True)

## 获取请求的各个token时延和迁移时间

In [9]:
import re
def plot_single(ax, latencies):
    hist, bin_edges = np.histogram(latencies, bins=50)
    cumsum = np.cumsum(hist)
    p50 = np.percentile(latencies, 50)
    p80 = np.percentile(latencies, 80)
    p95 = np.percentile(latencies, 95)
    p99 = np.percentile(latencies, 99)
    p999 = np.percentile(latencies, 99.9)
    print(f'p50:{p50},p80:{p80},p95:{p95},p99:{p99},p999:{p999}')
    ax.plot(bin_edges[1:], cumsum/np.sum(hist)*100, color='red')
    ax.axvline(p50, color='blue', linestyle='--', label='P50')
    ax.text(p50, ax.get_ylim()[0] + 0.05 * (ax.get_ylim()[1] - ax.get_ylim()[0]), f"{p50:.2f}", va='bottom', ha='right', color='blue')
    ax.axvline(p80, color='green', linestyle='--', label='P80')
    ax.text(p80, ax.get_ylim()[0] + 0.10 * (ax.get_ylim()[1] - ax.get_ylim()[0]), f"{p80:.2f}", va='bottom', ha='right', color='green')
    ax.axvline(p95, color='orange', linestyle='--', label='P95')
    ax.text(p95, ax.get_ylim()[0] + 0.15 * (ax.get_ylim()[1] - ax.get_ylim()[0]), f"{p95:.2f}", va='bottom', ha='right', color='orange')
    ax.axvline(p99, color='purple', linestyle='--', label='P99')
    ax.text(p99, ax.get_ylim()[0] + 0.20 * (ax.get_ylim()[1] - ax.get_ylim()[0]), f"{p99:.2f}", va='bottom', ha='right', color='purple')
    ax.axvline(p999, color='gray', linestyle='--', label='P99.9')
    ax.text(p999, ax.get_ylim()[0] + 0.25 * (ax.get_ylim()[1] - ax.get_ylim()[0]), f"{p999:.2f}", va='bottom', ha='right', color='gray')
    mean = np.mean(latencies)
    mean_value = bin_edges[:-1][np.where(bin_edges[:-1] <= mean)][-1]
    mean_percentage = cumsum[np.where(bin_edges[:-1] <= mean)][-1] / np.sum(hist) * 100
    ax.axvline(mean_value, color='black', linestyle='-', label='mean={:.2f}'.format(mean))
    ax.text(mean_value, mean_percentage, f"{mean_percentage:.2f}", va='bottom', ha='right', color='black')
    ax.legend(loc='upper right')
    ax.set_ylabel('Cumulative Percentage(%)')
def extract_migration_times(path):
    # 检查文件是否存在
    if not os.path.isfile(path):
        print(f"File {path} does not exist.")
        return {}
    # 初始化结果字典
    migration_times = {}

    # 正则表达式模式（保持不变）
    pattern = r'migrate request \[(.*?)\].*?cost: (\d+\.\d+) ms'

    # 逐行读取文件（自动处理大文件）
    with open(path, 'r', encoding='utf-8') as file:
        for line in file:
            # 直接处理每行（更高效）
            line = line.strip()  # 移除首尾空白字符
            if 'migrate done' in line and 'cost:' in line:
                match = re.search(pattern, line)
                if match:
                    # 提取请求ID和时间（逻辑不变）
                    ids_str = match.group(1)
                    time = float(match.group(2))
                    request_ids = [req_id.strip("'") for req_id in ids_str.split(', ')]
                    # 更新字典
                    for req_id in request_ids:
                        migration_times[req_id] = time
    return migration_times

def extract_migration_waiting_times(log_file_path, is_plot=False):
    # 定义一个字典，用于存储请求 ID 对应的时间戳
    request_timestamps = {}
    migrate_waiting_times = []
    # 打开日志文件并逐行读取
    with open(log_file_path, 'r') as file:
        for line in file:
            # 检查行是否包含 engine_step_timestamp_end 或 _migrate_out_one_request start
            if "engine_step_timestamp_end" in line or "_migrate_out_one_request start" in line \
                  or "MigrationStatus.ABORTED_DST, timestamps" in line \
                    or "MigrationStatus.ABORTED_SRC, timestamps" in line :
                # 使用正则表达式提取请求 ID
                request_id_match = re.search(r'[0-9a-f]{32}', line)
                # 使用正则表达式提取时间戳
                timestamp_match = re.search(r'timestamps: \d+\.\d+', line)

                if request_id_match and timestamp_match:
                    request_id = request_id_match.group()
                    timestamp = float(timestamp_match.group().split(":")[1])

                    # 如果请求 ID 不在字典中，则初始化一个条目
                    if request_id not in request_timestamps:
                        request_timestamps[request_id] = {
                            "engine_step_timestamp_end": None,
                            "_migrate_out_one_request start": None,
                            "migrate_start_count": 0,
                            "ABORTED_DST_count":0,
                            "ABORTED_SRC_count":0,
                        }

                    # 根据日志行内容更新对应的时间戳
                    if "engine_step_timestamp_end" in line:
                        request_timestamps[request_id]["engine_step_timestamp_end"] = timestamp
                    elif "_migrate_out_one_request start" in line:
                        request_timestamps[request_id]["_migrate_out_one_request start"] = timestamp
                        request_timestamps[request_id]["migrate_start_count"] += 1
                    elif "MigrationStatus.ABORTED_DST, timestamps" in line:
                        request_timestamps[request_id]["ABORTED_DST_count"] += 1
                    elif "MigrationStatus.ABORTED_SRC, timestamps" in line:
                        request_timestamps[request_id]["ABORTED_SRC_count"] += 1
                        
                    if request_timestamps[request_id]["_migrate_out_one_request start"] is not None and request_timestamps[request_id]["engine_step_timestamp_end"] is not None:
                        request_timestamps[request_id]["migrate_waiting_time"] = (request_timestamps[request_id]["_migrate_out_one_request start"] - request_timestamps[request_id]["engine_step_timestamp_end"]) *1000
                        request_timestamps[request_id]["migrate_waiting_time"] = max(0, request_timestamps[request_id]["migrate_waiting_time"])
                        migrate_waiting_times.append(request_timestamps[request_id]["migrate_waiting_time"])
    
    if is_plot and len(migrate_waiting_times) > 0:
        fig, (ax) = plt.subplots(1, 1, figsize=(7, 4.8))
        fig.suptitle(log_file_path, fontsize=14)
        plot_single(ax, migrate_waiting_times)
        
    return request_timestamps


In [12]:

def json_to_decode_latencies(json_file, csv_file, log_file=None):
    with open(json_file, 'r') as f:
        data = json.load(f)
    # 提取指定字段
    token_latencies_list = data[0]['all_decode_token_latencies']
    response_lens = data[0]['request_lens']
    # token_latencies_list是一个list，response_lens保存了每个请求在token_latencies_list中对应的长度，基于response_lens将其转换为二维数组
    output_len_per_seq = max(response_lens)
    token_latencies = np.zeros((len(response_lens), output_len_per_seq))

    t=0
    for i in range(len(response_lens)):
        token_latencies[i][:response_lens[i]] = token_latencies_list[t:t+response_lens[i]]
        # assert token_latencies[i][0] != token_latencies[i][1], f"token_latencies[{i}][0]== token_latencies[{i}][1] = {token_latencies[i][0]} == {token_latencies[i][1]}"
        if token_latencies[i][0] == token_latencies[i][1]:
            print(f"token_latencies[{i}][0] == token_latencies[{i}][1] = {token_latencies[i][0]} == {token_latencies[i][1]}")
        t += response_lens[i]
    
    # 将数据转换为DataFrame，将request_ids作为index
    df = pd.DataFrame(token_latencies, index=data[0]['request_ids'])

    # 设置列名为token_id
    df.columns = [f'decode_token_{i+1}' for i in range(output_len_per_seq)]

    # 获取迁移时间
    if 'pdd' in json_file.split('/')[-1]:
        if log_file is None:
            # 将'/workspace/llm-serve/Llumnix/logs/l40-pdd--2/llama-2-7b/poisson/benchmark_pdd_tp1_1000_qps_4_prompt_len_1024_response_len_64_1_1_latency_info.json'转化为'/workspace/llm-serve/Llumnix/logs/l40-pdd--2/llama-2-7b/poisson/benchmark_pdd_tp1_1000_qps_4_1_1_prompt_len_1024_response_len_64_latency_info.json'
            p_d = json_file[-22:-17]    # _1_1_
            tmp = json_file[:-22].split('qps_')
            tmp = tmp[0] + 'qps_' + tmp[1][0] + p_d + tmp[1][2:] + json_file[-17:]
            log_path = tmp.replace('_latency_info.json', '.log')
            # 将最后一个'benchmark'替换为'serve'
            if 'benchmark' in log_path:
                log_path = re.sub(r'benchmark(?!.*benchmark)', 'serve', log_path)
            print(log_path)
            # log_path = log_path.replace('benchmark', 'serve')
        else:
            log_path = log_file
        
        migration_times = extract_migration_times(log_path)
        migration_waiting_times = extract_migration_waiting_times(log_path)
        # print(log_path,len(migration_times))

        # 在df中添加一列，列名为migration_time，放在第一列
        df.insert(0, 'migration_time', 0.0)
        df.insert(0, 'migration_waiting_time', 0.0)
        # 遍历df的index，获取对应的migration_time
        for index in df.index:
            # 如果index在migration_times中，则将对应的值赋值给df['migration_time']
            # print(index, index in migration_times)
            if index in migration_times:
                df.at[index, 'migration_time'] = migration_times[index]

            if index in migration_waiting_times and "migrate_waiting_time" in migration_waiting_times[index]:
                # print(migration_waiting_times[index])
                df.at[index, 'migration_waiting_time'] = migration_waiting_times[index]['migrate_waiting_time']
        # 绘制迁移时间和迁移等待时间的直方图
        if len(df['migration_time']) > 0:
            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 4.8))
            fig.suptitle(log_file, fontsize=14)
            plot_single(ax1, df['migration_time'])
            ax1.set_title('Migration Time Distribution')
            plot_single(ax2, df['migration_waiting_time'])
            ax2.set_title('Migration Waiting Time Distribution')
            plt.tight_layout()
            plt.savefig(json_file.replace('.json', '_migration_times.png'))
            plt.close(fig)

    # print(csv_file)
    df.insert(0, 'prefill_time', data[0]['prefill_token_latencies'])
    # 保存为csv文件
    df.to_csv(csv_file, index=True)

# glob.glob(os.path.join(dir, '*.json'))
filenames = [
    # '/workspace/llm-serve/Llumnix/benchmark_test/logs/l40-pdd-hetero/llama-7b/poisson/benchmark_pdd_2000_qps_6_1,1_2_latency_info.json',
    '/workspace/llm-serve/Llumnix/benchmark_test/logs/A6000-2-concurrency-1-512-256/llama-13b/poisson/benchmark_pdd_2000_qps_2_1,1_2_latency_info.json'
]
log_filenames = [
    '/workspace/llm-serve/Llumnix/benchmark_test/logs/A6000-2-concurrency-1-512-256/llama-13b/poisson/serve_pdd_2000_qps_2_1,1_2.log'
]
for filename, log_file in zip(filenames, log_filenames):
    
    output_path = filename.replace('latency_info.json', 'all_decode_token_latencies.csv')
    if os.path.exists(output_path):
        continue
    print(filename)
    json_to_decode_latencies(filename, output_path, log_file=log_file)

/workspace/llm-serve/Llumnix/benchmark_test/logs/A6000-2-concurrency-1-512-256/llama-13b/poisson/benchmark_pdd_2000_qps_2_1,1_2_latency_info.json


findfont: Generic family 'sans-serif' not found because none of the following families were found: SimHei
findfont: Generic family 'sans-serif' not found because none of the following families were found: SimHei
findfont: Generic family 'sans-serif' not found because none of the following families were found: SimHei
findfont: Generic family 'sans-serif' not found because none of the following families were found: SimHei
findfont: Generic family 'sans-serif' not found because none of the following families were found: SimHei
findfont: Generic family 'sans-serif' not found because none of the following families were found: SimHei
findfont: Generic family 'sans-serif' not found because none of the following families were found: SimHei
findfont: Generic family 'sans-serif' not found because none of the following families were found: SimHei
findfont: Generic family 'sans-serif' not found because none of the following families were found: SimHei
findfont: Generic family 'sans-serif' not foun

p50:484.1958284378052,p80:561.8762016296387,p95:632.5780749320984,p99:711.9149589538574,p999:787.7853927612405
p50:493.11530590057373,p80:2737.5864505767827,p95:55055.04826307295,p99:922215.474114418,p999:957594.9445989142


findfont: Generic family 'sans-serif' not found because none of the following families were found: SimHei
findfont: Generic family 'sans-serif' not found because none of the following families were found: SimHei
findfont: Generic family 'sans-serif' not found because none of the following families were found: SimHei
findfont: Generic family 'sans-serif' not found because none of the following families were found: SimHei
findfont: Generic family 'sans-serif' not found because none of the following families were found: SimHei
findfont: Generic family 'sans-serif' not found because none of the following families were found: SimHei
findfont: Generic family 'sans-serif' not found because none of the following families were found: SimHei
findfont: Generic family 'sans-serif' not found because none of the following families were found: SimHei
findfont: Generic family 'sans-serif' not found because none of the following families were found: SimHei
findfont: Generic family 'sans-serif' not foun

## 获取迁移相关信息

In [ ]:
def extract_migration_info(path, file_output=None, tp_hetero=False, verbose=False):
    '''
    migration_info[req_id] = {
                            "blocks": blocks,
                            "time_ms": migrate time,
                            "speed_blocks_per_s": speed
                            "migrate_waiting_time": migrate_waiting_time(ms)
                        }
    '''
    print(f"Processing log file: {path}")
    if file_output is None:
        file_output = open('/tmp/benchmark_tmp', "w")
    file_output.write(f"Processing log file: {path}")

    # 检查文件是否存在
    if not os.path.isfile(path):
        file_output.write(f"File {path} does not exist.")
        return {}

    migration_info = {}
    reject_migrate_in_count = 0
    reject_migrate_out_count = 0
    # 示例： Instance ... migrate done, migrate request ['494c45676def4572986621d1afbc337f'], migration status: MigrationStatus.FINISHED, len: 7 blocks, cost: 240.65113067626953 ms
    # 正确的正则表达式应为：
    pattern = r"migrate request \[(.*?)\].*?len: (\d+) blocks,.*?cost: ([\d\.]+) ms"
    count = 0
    finished_flag = False
    if not tp_hetero:
        finished_str = "Entrypoints num_finished_requests: 503"
    else:
        finished_str = "Entrypoints num_finished_requests: 671"
    # 逐行读取文件（自动处理大文件）
    with open(path, 'r', encoding='utf-8') as file:
        for line in file:
            line = line.strip()
            # if "Error" in line:
            #     print(f"Error in line:{line}")
            
            if finished_str in line:
                finished_flag = True
            if 'reject new migrate out' in line:
                reject_migrate_out_count += 1
            if 'reject new migrate in' in line:
                reject_migrate_in_count += 1
            # 获取迁移时间和速度
            if 'migrate done' in line and 'cost:' in line:
                if count < 10:
                    count += 1
                    continue
                match = re.search(pattern, line)
                if match:
                    ids_str = match.group(1)
                    blocks = int(match.group(2))
                    time = float(match.group(3))
                    speed = blocks / time * 1000 if time > 0 else 0  # blocks/ms -> blocks/s
                    request_ids = [req_id.strip("'") for req_id in ids_str.split(', ')]
                    for req_id in request_ids:
                        if len(req_id) > 0:
                            assert req_id in migration_info, f"{req_id},{type(req_id)},{line}"
                            migration_info[req_id]["blocks"] = blocks
                            migration_info[req_id]["time_ms"] = time
                            migration_info[req_id]["speed_blocks_per_s"] = speed

            if "engine_step_timestamp_end" in line or "_migrate_out_one_request start" in line \
                  or "MigrationStatus.ABORTED_DST, timestamps" in line \
                    or "MigrationStatus.ABORTED_SRC, timestamps" in line :
                # 使用正则表达式提取请求 ID
                request_id_match = re.search(r'[0-9a-f]{32}', line)
                # 使用正则表达式提取时间戳
                timestamp_match = re.search(r'timestamps: \d+\.\d+', line)

                if request_id_match and timestamp_match:
                    request_id = request_id_match.group()
                    timestamp = float(timestamp_match.group().split(":")[1])

                    # 如果请求 ID 不在字典中，则初始化一个条目
                    if request_id not in migration_info:
                        migration_info[request_id] = {
                            "blocks": 0,
                            "time_ms":  0.0,
                            "speed_blocks_per_s": 0.0,
                            "engine_step_timestamp_end": None,
                            "migrate_out_one_request_start": None,
                            "migrate_start_count": 0,
                            "ABORTED_DST_count":0,
                            "ABORTED_SRC_count":0,
                        }

                    # 根据日志行内容更新对应的时间戳
                    if "engine_step_timestamp_end" in line:
                        migration_info[request_id]["engine_step_timestamp_end"] = timestamp
                    if migration_info[request_id]["time_ms"] == 0.0:
                        if "_migrate_out_one_request start" in line:
                            migration_info[request_id]["migrate_out_one_request_start"] = timestamp
                            migration_info[request_id]["migrate_start_count"] += 1
                        elif "MigrationStatus.ABORTED_DST, timestamps" in line:
                            migration_info[request_id]["ABORTED_DST_count"] += 1
                        elif "MigrationStatus.ABORTED_SRC, timestamps" in line:
                            migration_info[request_id]["ABORTED_SRC_count"] += 1
                            
                        if migration_info[request_id]["migrate_out_one_request_start"] is not None and migration_info[request_id]["engine_step_timestamp_end"] is not None:
                            migration_info[request_id]["migrate_waiting_time"] = (migration_info[request_id]["migrate_out_one_request_start"] - migration_info[request_id]["engine_step_timestamp_end"]) *1000
                            migration_info[request_id]["migrate_waiting_time"] = max(0, migration_info[request_id]["migrate_waiting_time"])
                            # migrate_waiting_times.append(migration_info[request_id]["migrate_waiting_time"])
                    else:
                        # print("not first migration")
                        pass

    fail_req_id = set()
    for req_id, info in migration_info.items():
        if 'migrate_waiting_time' not in migration_info[req_id]:
            fail_req_id.add(req_id)
            if verbose:
                file_output.write(f'fail req_id:{req_id}, no migrate_waiting_time, {migration_info[req_id]}')
        else:
            if migration_info[req_id]["migrate_waiting_time"] > 1000:
                pass
                # file_output.write(f'req_id:{req_id},migrate_waiting_time:{migration_info[req_id]["migrate_waiting_time"]},ABORTED_DST_count:{migration_info[req_id]["ABORTED_DST_count"]}')
        if migration_info[req_id]['time_ms'] == 0.0:
            fail_req_id.add(req_id)
            if verbose:
                file_output.write(f'fail req_id:{req_id}, no migrate_time, {migration_info[req_id]}')
    for req_id in fail_req_id:
        del migration_info[req_id]
    
    if migration_info:
        avg_speed = mean(info['speed_blocks_per_s'] for info in migration_info.values())
        avg_migration_time = mean(info['time_ms'] for info in migration_info.values())
        avg_migrate_waiting_time = mean(info["migrate_waiting_time"] for info in migration_info.values())
        avg_migration_count = mean(info['migrate_start_count'] for info in migration_info.values())
        avg_migration_aborted_dst_count = mean(info['ABORTED_DST_count'] for info in migration_info.values())

        file_output.write(f"fail req num(lose msg): {len(fail_req_id)}, finished_flag:{finished_flag},finished_str:{finished_str}")
        file_output.write(f"Average migration speed: {avg_speed:.2f} blocks/s")
        file_output.write(f"Average migration time: {avg_migration_time:.2f} ms")
        file_output.write(f'Average migration waiting time: {avg_migrate_waiting_time:.2f} ms')
        file_output.write(f"Average migration count: {avg_migration_count}")
        file_output.write(f"Average migration ABORTED_DST count: {avg_migration_aborted_dst_count}")
        file_output.write(f"Sum migration ABORTED_DST count: {avg_migration_aborted_dst_count*len(migration_info)}")
        file_output.write(f"reject_migrate_out_count:{reject_migrate_out_count}")
        file_output.write(f"reject_migrate_in_count:{reject_migrate_in_count}")
        file_output.write(f"max block num : {max(info['blocks'] for info in migration_info.values())}")
        # if avg_migration_count > avg_migration_aborted_dst_count + 
        for req_id, info in migration_info.items():
            info['avg_speed_blocks_per_s'] = avg_speed
    else:
        file_output.write("No migration information found.")

    file_output.write("\n")
    # assert finished_flag
    res = {
        'avg_speed': avg_speed,
        'avg_migration_time': avg_migration_time,
        'avg_migrate_waiting_time': avg_migrate_waiting_time,
        'avg_migration_count': avg_migration_count,
        'avg_migration_aborted_dst_count': avg_migration_aborted_dst_count,
        'sum_migration_aborted_dst_count': avg_migration_aborted_dst_count*len(migration_info),
        'reject_migrate_out_count': reject_migrate_out_count,
        'reject_migrate_in_count': reject_migrate_in_count,
    }
    return res
extract_migration_info('/workspace/llm-serve/Llumnix/benchmark_test/logs/A6000-test-concurrency-4-128-256/llama-13b/poisson/serve_pdd_1000_qps_16_1_1,1,1.log',
                         )

Processing log file: /workspace/llm-serve/Llumnix/benchmark_test/logs/A6000-test-concurrency-4-128-256/llama-13b/poisson/serve_pdd_1000_qps_16_1_1,1,1.log


{'avg_speed': 837.9260462293722,
 'avg_migration_time': 127.88592386245728,
 'avg_migrate_waiting_time': 26897.599772691727,
 'avg_migration_count': 1,
 'avg_migration_aborted_dst_count': 0,
 'sum_migration_aborted_dst_count': 0,
 'reject_migrate_out_count': 1007,
 'reject_migrate_in_count': 0}

In [ ]:

path_tmp = 'A6000-2-formal2'
model = 'llama-13b'
max_concurrent = 16
instance_num = 4
qps = [2,4]
# log_paths = [
#     '/workspace/llm-serve/Llumnix/benchmark_test/logs/A6000-2-formal-concurrency-8-pdd-4/llama-7b/poisson/serve_pdd_tp1_2000_qps_2_1_3.log'
# ]
output_file = f"output-{path_tmp}-{model}-max_concurrent_{max_concurrent}.txt"
if os.path.isfile(output_file):
    os.remove(output_file)
with open(output_file, "w") as file:
    for q in qps:
        for prefill_num in range(1,instance_num):
            decode_num = instance_num-prefill_num
            log_path = f'/workspace/llm-serve/Llumnix/benchmark_test/logs/{path_tmp}-concurrency-{max_concurrent}-pdd-4/{model}/poisson/serve_pdd_tp1_2000_qps_{q}_{prefill_num}_{decode_num}.log'
    # for log_path in log_paths:
            
            # 调用函数提取迁移速度
            extract_migration_info(log_path, file)
        log_path = f'/workspace/llm-serve/Llumnix/benchmark_test/logs/{path_tmp}-pdd-hetero-concurrency-{max_concurrent}/{model}/poisson/serve_pdd_2000_qps_{q}_1,1_2.log'
        extract_migration_info(log_path, file, True)


## 获取较为宏观的信息(时延和迁移相关信息)

In [ ]:
def get_lantency(json_file):
    if not os.path.isfile(json_file):
        print(f"File {json_file} does not exist.")
        return None
    print(f'Process file: {json_file}')
    try:
        with open(json_file, 'r') as f:
            data = json.load(f)
    except Exception as e:
        print(f'error:{str(e)}')
    assert len(data) == 1, "Expected data to contain only one entry"
    latencies = data[0]
    req_latencies, prefill_latencies, decode_latencies = latencies['request_latencies'], latencies['prefill_token_latencies'], latencies['decode_token_latencies']
    return round(mean(req_latencies), 2), round(mean(prefill_latencies), 2), round(mean(decode_latencies), 2)

path_tmp = 'A6000-2-formal2'
model = 'llama-13b'
concurrencies = [1,2,4,8,16]
instance_num = 4
qps = [2,4]
json_file = '/workspace/llm-serve/Llumnix/benchmark_test/logs/A6000-2-formal2-concurrency-1-pdd-4/llama-13b/poisson/benchmark_4_tp1_2000_qps_2_latency_info.json'
# for concurrency in concurrencies:
#     for q in qps:
#         json_file = f'/workspace/llm-serve/Llumnix/benchmark_test/logs/{path_tmp}-concurrency-{concurrency}-pdd-{instance_num}/{model}/poisson/benchmark_{instance_num}_tp1_2000_qps_{q}_latency_info.json'
#         print(get_lantency(json_file))
#         for prefill_num in range(1,instance_num):
#             decode_num = instance_num-prefill_num
#             json_file = f'/workspace/llm-serve/Llumnix/benchmark_test/logs/{path_tmp}-concurrency-{concurrency}-pdd-{instance_num}/{model}/poisson/benchmark_pdd_tp1_2000_qps_{q}_{prefill_num}_{decode_num}_latency_info.json'
#             print(get_lantency(json_file))
#         json_file = f'/workspace/llm-serve/Llumnix/benchmark_test/logs/{path_tmp}-pdd-hetero-concurrency-{concurrency}/{model}/poisson/benchmark_pdd_2000_qps_{q}_1,1_2_latency_info.json'
#         print(get_lantency(json_file))

In [ ]:
import pandas as pd
import pickle
def get_label(prefill_tps, decode_tps, is_pd=True):
    if not is_pd:
        res = ",".join(str(x) for x in prefill_tps)
    else:
        res = ",".join(str(x) for x in prefill_tps) + '-' + ",".join(str(x) for x in decode_tps)
    return res

path_tmp = 'A6000-2-formal2'
model = 'llama-7b'
instance_num = 4
if model == 'llama-7b':
    concurrencies = [1,2,4,8,16]
    qps = [2,4,6,8,10,12]
elif model == 'llama-13b':
    concurrencies = [1,2,4,8,16]
    qps = [1,2,4,]
results = {}            # {(qps, concurrency): (req, prefill, decode)}
migration_infos = {}    # {(qps, concurrency): migration_info}
labels = []             # 延迟的label，用于区分不同资源分配下的latency (3*(instance_num+1)个)
migration_labels = []   # 迁移数据的label，instance_num个，如1-1,1,1、1,1-1,1、1,1,1-1、1,1-2)
cache_file = f'latency_results_cache-{path_tmp}-{model}-{concurrencies}-{qps}.pkl'
result_file = f'latency_results-{path_tmp}-{model}-{concurrencies}-{qps}.xlsx'
migration_infos_file = f'migration_infos-{path_tmp}-{model}-{concurrencies}-{qps}.xlsx'

# 如果有缓存文件则直接加载
if os.path.exists(cache_file):
    with open(cache_file, 'rb') as f:
        data_res = pickle.load(f)
        print(data_res.keys())
        results = data_res['results']
        label = data_res['labels']
        migration_infos = data_res['migration_infos'] if 'migration_infos' in data_res else {}
        migration_labels = data_res['migration_labels'] if 'migration_labels' in data_res else []

for q in qps:
    for concurrency in concurrencies:
        if (q, concurrency) in results:
            continue
        result = []
        json_file = f'/workspace/llm-serve/Llumnix/benchmark_test/logs/{path_tmp}-concurrency-{concurrency}-pdd-{instance_num}/{model}/poisson/benchmark_{instance_num}_tp1_2000_qps_{q}_latency_info.json'
        result.extend(get_lantency(json_file))
        
        for prefill_num in range(1,instance_num):
            decode_num = instance_num-prefill_num
            json_file = f'/workspace/llm-serve/Llumnix/benchmark_test/logs/{path_tmp}-concurrency-{concurrency}-pdd-{instance_num}/{model}/poisson/benchmark_pdd_tp1_2000_qps_{q}_{prefill_num}_{decode_num}_latency_info.json'
            result.extend(get_lantency(json_file))
            labels.extend(get_label([1]*prefill_num, [1]*decode_num))
        json_file = f'/workspace/llm-serve/Llumnix/benchmark_test/logs/{path_tmp}-pdd-hetero-concurrency-{concurrency}/{model}/poisson/benchmark_pdd_2000_qps_{q}_1,1_2_latency_info.json'
        result.extend(get_lantency(json_file))
        results[(q, concurrency)] = result
        # 保存结果到本地
        with open(cache_file, 'wb') as f:
            data_res = {}
            data_res['results'] = results
            data_res['labels'] = labels
            data_res['migration_infos'] = migration_infos
            data_res['migration_labels'] = migration_labels
            pickle.dump(data_res, f)
            
for q in qps:
    for concurrency in concurrencies:
        if (q, concurrency) in migration_infos:
            continue
        migration_info = {}
        for prefill_num in range(1,instance_num):
            decode_num = instance_num - prefill_num
            log_path = f'/workspace/llm-serve/Llumnix/benchmark_test/logs/{path_tmp}-concurrency-{concurrency}-pdd-4/{model}/poisson/serve_pdd_tp1_2000_qps_{q}_{prefill_num}_{decode_num}.log'

            # 调用函数提取迁移信息
            info = extract_migration_info(log_path, None)
            migration_info[get_label([1]*prefill_num, [1]*decode_num)] = info
        log_path = f'/workspace/llm-serve/Llumnix/benchmark_test/logs/{path_tmp}-pdd-hetero-concurrency-{concurrency}/{model}/poisson/serve_pdd_2000_qps_{q}_1,1_2.log'
        info = extract_migration_info(log_path, None, True)
        migration_info[get_label([1]*2, [2]*1)] = info
            
        migration_infos[(q, concurrency)] = migration_info
labels=[]
labels.extend([get_label([1]*instance_num, 0, False)]*3)
for prefill_num in range(1,instance_num):
    decode_num = instance_num-prefill_num
    labels.extend([get_label([1]*prefill_num, [1]*decode_num)]*3)
labels.extend([get_label([1]*2, [2]*1)]*3)

migration_labels=[]
for prefill_num in range(1,instance_num):
    decode_num = instance_num-prefill_num
    migration_labels.extend([get_label([1]*prefill_num, [1]*decode_num)])
migration_labels.extend([get_label([1]*2, [2]*1)])

# 保存结果到本地
with open(cache_file, 'wb') as f:
    data_res = {}
    data_res['results'] = results
    data_res['labels'] = labels
    
    for q in qps:
        for  concurrency in concurrencies:
            if '1,1-1' in migration_infos[(q, concurrency)] or '1-1' in migration_infos[(q, concurrency)]:
                # migration_infos[(q, concurrency)]['1-1,1,1'] = migration_infos[(q, concurrency)]['1-1']
                # migration_infos[(q, concurrency)]['1,1-1,1'] = migration_infos[(q, concurrency)]['1,1-1']
                del migration_infos[(q, concurrency)]['1,1-1']
                del migration_infos[(q, concurrency)]['1-1']
    data_res['migration_infos'] = migration_infos
    data_res['migration_labels'] = migration_labels
    pickle.dump(data_res, f)
print(len(labels))
with pd.ExcelWriter(result_file) as writer:
    for q in qps:
        # 构造以 concurrency 为行，latency 为列的 DataFrame
        data = [results[(q, concurrency)] for concurrency in concurrencies]
        df_qps = pd.DataFrame(data, index=concurrencies, columns=labels)
        df_qps.index.name = 'concurrency'
        df_qps.to_excel(writer, sheet_name=f'qps_{q}')
metrics = ['avg_speed',
        'avg_migration_time',
        'avg_migrate_waiting_time',
        'avg_migration_count',
        'avg_migration_aborted_dst_count',
        'sum_migration_aborted_dst_count',
        'reject_migrate_out_count',
        'reject_migrate_in_count',
    ]

with pd.ExcelWriter(migration_infos_file) as writer:
    for metric in metrics:
        for q in qps:
            # 构造以 concurrency 为行，latency 为列的 DataFrame
            data = [migration_infos[(q, concurrency)] for concurrency in concurrencies]
            data_all = []
            for single_data in data:
                data_cur_concurrency = []
                
                for label in migration_labels:
                    data_cur_concurrency.append(single_data[label][metric])
                data_all.append(data_cur_concurrency)
            df_qps = pd.DataFrame(data_all, index=concurrencies, columns=migration_labels)
            df_qps.index.name = 'concurrency'
            df_qps.to_excel(writer, sheet_name=f'qps_{q}_{metric}')

In [ ]:
def get_step_lantency(json_file):
    if not os.path.isfile(json_file):
        print(f"File {json_file} does not exist.")
        return None
    print(f'Process file: {json_file}')
    try:
        with open(json_file, 'r') as f:
            data = json.load(f)
    except Exception as e:
        print(f'error:{str(e)}')
    assert len(data) == 1, "Expected data to contain only one entry"
    per_token_latency_breakdown_list = data[0]['per_token_latency_breakdown_list']
    prefill_waiting_time = [(per_token_latency_breakdown_list[i][0]['engine_step_timestamp_begin'] - per_token_latency_breakdown_list[i][0]['engine_add_request_timestamp'])*1000
                             for i in range(len(per_token_latency_breakdown_list))]
    engine_step_latency_prefill = [per_token_latency_breakdown_list[i][0]['engine_step_latency'] for i in range(len(per_token_latency_breakdown_list))]
    engine_step_latency_decode = [
        mean([token['engine_step_latency'] for token in per_token_latency_breakdown_list[i][1:]])
        if len(per_token_latency_breakdown_list[i][1:]) > 0 else None
        for i in range(len(per_token_latency_breakdown_list))
    ]

    return round(mean(prefill_waiting_time), 2), round(mean(engine_step_latency_prefill), 2), round(mean([x for x in engine_step_latency_decode if x is not None]), 2)
json_file = '/workspace/llm-serve/Llumnix/benchmark_test/logs/A6000-2-formal2-pdd-hetero-concurrency-4/llama-13b/poisson/benchmark_pdd_2000_qps_1_1,1_2_latency_info.json'
get_step_lantency(json_file)

## 汇总
上文提到的大多数信息，均可以通过main.py获得

### A6000-2-formal2

In [12]:
from main import LogAnalysis

# 提取信息
model = 'llama-13b' # 'llama-7b'
analysis = LogAnalysis(model)
analysis.get_all_msg()  # 首次需要
analysis.get_all_msg_updata_instance_metric()
# 结果保存在 analysis.results 中，[concurrency][qps][label]

# data_to_show = {
#     'concurrency': str(16),
#     'qps': str(6),
#     'labels': ["1,1-1,1",  "1,1-2"]# ["1,1,1,1", "1-1,1,1", "1,1-1,1", "1,1,1-1", "1,1-2"]
# }

def get_value(key, fix_fields, change_field_value):
    if key in fix_fields:
        return str(fix_fields[key])
    else:
        return str(change_field_value)
fix_fields = {
    'qps': str(2),
    # 'labels': "1,1,1,1"
    'concurrency': 4 # [1,2,4,8,16]
}
change_field = {
    'labels': ["1,1,1,1", "1-1,1,1", "1,1-1,1", "1,1,1-1", "1,1-2"]
}

assert len(change_field) == 1

data = []
key = list(change_field.keys())[0]
values = list(change_field.values())[0]
for v in values:
    concurrency = get_value('concurrency', fix_fields, v)
    qps = get_value('qps', fix_fields, v)
    label = get_value('labels', fix_fields, v)
    data.append(analysis.results[concurrency][qps][label])
# 显示所有列
pd.set_option('display.max_columns', None)
# 显示所有行
pd.set_option('display.max_rows', None)
# 设置每列宽度（可选）
pd.set_option('display.max_colwidth', None)
df = pd.DataFrame(data, index=[f'{key}_{v}' for v in values])
df

[LogAnalysis] exist cache_file:results/latency_results_cache-A6000-2-formal2-llama-13b-[1, 2, 4, 8, 16]-[1, 2, 4].json
[get_all_msg_qps_updata_instance_metric] concurrency:1, qps:1
[get_instance_metrics] Processing file: /workspace/llm-serve/Llumnix/benchmark_test/logs/A6000-2-formal2-concurrency-1-pdd-4/llama-13b/poisson/serve_4_tp1_2000_qps_1_instance.csv


KeyboardInterrupt: 

### A6000-2

In [2]:
from benchmark_test.analysis.main import LogAnalysis_new
import pandas as pd

def get_value(key, fix_fields, change_field_value):
    if key in fix_fields:
        return str(fix_fields[key])
    else:
        return str(change_field_value)


#### llama-7b

In [8]:

model = 'llama-7b' # 'llama-7b'
instance_deploy_msg = [     # (prefill_tps, decode_tps)
    ([1,1,1,1],[]),([1],[1,1,1]),([1,1,],[1,1]),([1,1],[2]),([1,1,1],[]),([1],[2])
]
analysis = LogAnalysis_new(model,[4],[1,2,4,8],instance_deploy_msg,'256-256')
analysis.get_all_msg()  # 首次需要
# analysis.get_all_msg_updata_instance_metric()
metrics = [
        ["request_time", "prefill_time", "decode_time",],
        'prefill_bs',
        'prefill_all_time_bs'
]
analysis.translate_to_excel_according_metrics(metrics)

fix_fields = {
    'qps': str(4),
    # 'labels': "1,1,1,1"
    'concurrency': 4 # [1,2,4,8,16]
}
change_field = {
    'labels': ["1,1,1,1", "1-1,1,1", "1,1-1,1", "1,1-2"]
}

assert len(change_field) == 1

data = []
key = list(change_field.keys())[0]
values = list(change_field.values())[0]
for v in values:
    concurrency = get_value('concurrency', fix_fields, v)
    qps = get_value('qps', fix_fields, v)
    label = get_value('labels', fix_fields, v)
    data.append(analysis.results[concurrency][qps][label])
# 显示所有列
pd.set_option('display.max_columns', None)
# 显示所有行
pd.set_option('display.max_rows', None)
# 设置每列宽度（可选）
pd.set_option('display.max_colwidth', None)
df = pd.DataFrame(data, index=[f'{key}_{v}' for v in values])
df

[LogAnalysis] cache_file:results/results_cache-A6000-2-llama-7b-256-256-[4]-[1, 2, 4, 8].json
[LogAnalysis] exist cache_file:results/results_cache-A6000-2-llama-7b-256-256-[4]-[1, 2, 4, 8].json
[translate_to_excel_according_metrics] output_path:results/results_cache-A6000-2-llama-7b-256-256-[4]-[1, 2, 4, 8].xlsx


/root/anaconda3/envs/llumnix/lib/python3.10/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


,request_time,prefill_time,decode_time,906c21d3dc38401db795b3079aed59f7,926f57a635374f8fb691ec91e5ffab3c,d54fe8a80a924828908cdf02f50ee143,f13153d4db9d4848b675eeb6b563f145,avg_speed,avg_migration_time,avg_migrate_waiting_time,avg_migration_count,avg_migration_aborted_dst_count,sum_migration_aborted_dst_count,reject_migrate_out_count,reject_migrate_in_count,03ba923b6c2a4838979309e44b1efb61,1b646a012af442fe905223b44737550c,1eab4b77c7a6424aab1c6be68a59ae64,77f083c52a5f46909a7869b5a3394e4b,a61a2215974b4531af87d847aa494438,ac05ff0f5f5b47ce8f99c325e9006416,cc340ce0548946b2b1865e331b949197,f8702c6d2cff4b90bbad8b1666ea2e5c,95712557c26041dd8fbeb41ab62b2bb1,ba763691d77a4562a82067ff9ba513b7,deb54204505c47ab9bada1e7155fef8e
"labels_1,1,1,1",8.2830,116.1564,31.8642,"{'decode_bs': 8.3276, 'decode_ratio': 0.9504, 'gpu_cache_usage': 0.053407, 'num_running_requests': 8.318891, 'num_waiting_requests': 0, 'num_killed_requests': 0, 'sm_active': 0.649748, 'mofc': 0.955944, 'prefill_step_time': 50.199948, 'decode_step_time': 27.865354}","{'decode_bs': 8.2707, 'decode_ratio': 0.9476, 'gpu_cache_usage': 0.053059, 'num_running_requests': 8.264291, 'num_waiting_requests': 0, 'num_killed_requests': 0, 'sm_active': 0.647368, 'mofc': 0.953677, 'prefill_step_time': 50.270859, 'decode_step_time': 27.942159}","{'decode_bs': 8.2488, 'decode_ratio': 0.9508, 'gpu_cache_usage': 0.052928, 'num_running_requests': 8.24372, 'num_waiting_requests': 0, 'num_killed_requests': 0, 'sm_active': 0.65117, 'mofc': 0.95479, 'prefill_step_time': 50.361142, 'decode_step_time': 27.870577}","{'decode_bs': 8.1403, 'decode_ratio': 0.951, 'gpu_cache_usage': 0.05231, 'num_running_requests': 8.145629, 'num_waiting_requests': 0, 'num_killed_requests': 0, 'sm_active': 0.647449, 'mofc': 0.95656, 'prefill_step_time': 51.240848, 'decode_step_time': 27.787565}",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"labels_1-1,1,1",9.3212,110.7656,35.9488,NaN,NaN,NaN,NaN,105.2188,158.2392,16.6744,1.000,0.000,0.0,4.0,0.0,"{'inference_type': 'decode', 'decode_bs': 11.2942, 'decode_all_time_bs': 11.2843, 'gpu_cache_usage': 0.073352, 'num_running_requests': 11.284306, 'num_waiting_requests': 0, 'num_killed_requests': 0, 'sm_active': 0.588903, 'mofc': 0.972708, 'decode_step_time': 32.163728}","{'inference_type': 'prefill', 'prefill_bs': 269.3623, 'prefill_all_time_bs': 103.0561, 'gpu_cache_usage': 0.00629, 'num_running_requests': 0.654352, 'num_waiting_requests': 0, 'num_killed_requests': 0, 'sm_active': 0.118668, 'mofc': 0.183987, 'prefill_step_time': 58.8134}","{'inference_type': 'decode', 'decode_bs': 11.6142, 'decode_all_time_bs': 11.604, 'gpu_cache_usage': 0.075434, 'num_running_requests': 11.603969, 'num_waiting_requests': 0, 'num_killed_requests': 0, 'sm_active': 0.587986, 'mofc': 0.972344, 'decode_step_time': 32.412268}","{'inference_type': 'decode', 'decode_bs': 11.2225, 'decode_all_time_bs': 11.2127, 'gpu_cache_usage': 0.072904, 'num_running_requests': 11.212722, 'num_waiting_requests': 0, 'num_killed_requests': 0, 'sm_active': 0.590614, 'mofc': 0.973235, 'decode_step_time': 32.096298}",NaN,NaN,NaN,NaN,NaN,NaN,NaN
"labels_1,1-1,1",10.6336,100.6247,41.1375,NaN,NaN,NaN,NaN,112.6882,145.1357,17.6716,1.000,0.000,0.0,0.0,0.0,NaN,NaN,NaN,NaN,"{'inference_type': 'decode', 'decode_bs': 19.2499, 'decode_all_time_bs': 19.2332, 'gpu_cache_usage': 0.124794, 'num_running_requests': 19.233188, 'num_waiting_requests': 0, 'num_killed_requests': 0, 'sm_active': 0.552475, 'mofc': 0.973287, 'decode_step_time': 36.236437}","{'inference_type': 'prefill', 'prefill_bs': 262.3445, 'prefill_all_time_bs': 71.8642, 'gpu_cache_usage': 0.004033, 'num_running_requests': 0.436185, 'num_waiting_requests': 0, 'num_killed_requests': 0, 'sm_active': 0.042312, 'mofc': 0.249094, 'prefill_step_time': 56.244277}","{'inference_type': 'decode', 'decode_bs': 19.3847, 'decode_all_time_bs': 19.3657, 'gpu_cache_usage': 0.125695, 'num_running_requests': 19.365729, 'num_waiting_requests': 0, 'num_killed_requests

In [7]:

model = 'llama-7b' # 'llama-7b'
instance_deploy_msg = [     # (prefill_tps, decode_tps)
    ([1,1,1,1],[]),([1],[1,1,1]),([1,1,],[1,1]),([1,1],[2]),([1,1,1],[]),([1],[2])
]
analysis = LogAnalysis_new(model,[4],[1,2,4,8],instance_deploy_msg,'512-256')
analysis.get_all_msg()  # 首次需要
metrics = [
        ["prefill_step_time", "decode_step_time", 'avg_migration_time','avg_migrate_waiting_time'],
        ["mofc", "decode_bs", "decode_ratio", "sum_migration_aborted_dst_count"],
        ["mofc", "prefill-mofc", "decode-mofc",]
]
analysis.translate_to_excel_according_metrics(metrics, suffix="more_info")

[LogAnalysis] cache_file:results/results_cache-A6000-2-llama-7b-512-256-[4]-[1, 2, 4, 8].json
[LogAnalysis] exist cache_file:results/results_cache-A6000-2-llama-7b-512-256-[4]-[1, 2, 4, 8].json
[translate_to_excel_according_metrics] output_path:results/results_cache-A6000-2-llama-7b-512-256-[4]-[1, 2, 4, 8]_more_info.xlsx


/root/anaconda3/envs/llumnix/lib/python3.10/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")
/root/anaconda3/envs/llumnix/lib/python3.10/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")
/root/anaconda3/envs/llumnix/lib/python3.10/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


#### llama-13b

In [2]:

model = 'llama-13b' # 'llama-7b'
instance_deploy_msg = [     # (prefill_tps, decode_tps)
    ([1,1,1,1],[]),([1,1],[2]),([1,1,1],[]),([1],[2]),
]
for req_len in ['128-256','256-256','512-256',]:
    analysis = LogAnalysis_new(model,[2],[1,2,4,8],instance_deploy_msg,req_len)
    analysis.get_all_msg()  # 首次需要
    # analysis.get_all_msg_updata_instance_metric()
    metrics = [
            ["request_time", "prefill_time", "decode_time"],
            'prefill_bs',
            'prefill_all_time_bs'
    ]
    analysis.translate_to_excel_according_metrics(metrics)  # 将结果转换为Excel格式

fix_fields = {
    'qps': str(2),
    # 'labels': "1,1,1,1"
    'concurrency': 4 # [1,2,4,8,16]
}
change_field = {
    'labels': ["1,1,1,1", "1,1-2", "1,1,1", "1-2"]
}

assert len(change_field) == 1

data = []
key = list(change_field.keys())[0]
values = list(change_field.values())[0]
for v in values:
    concurrency = get_value('concurrency', fix_fields, v)
    qps = get_value('qps', fix_fields, v)
    label = get_value('labels', fix_fields, v)
    data.append(analysis.results[concurrency][qps][label])
# 显示所有列
pd.set_option('display.max_columns', None)
# 显示所有行
pd.set_option('display.max_rows', None)
# 设置每列宽度（可选）
pd.set_option('display.max_colwidth', None)
df = pd.DataFrame(data, index=[f'{key}_{v}' for v in values])
df

[LogAnalysis] cache_file:results/results_cache-A6000-2-llama-13b-128-256-[2]-[1, 2, 4, 8].json
[LogAnalysis] exist cache_file:results/results_cache-A6000-2-llama-13b-128-256-[2]-[1, 2, 4, 8].json
[translate_to_excel_according_metrics] output_path:results/results_cache-A6000-2-llama-13b-128-256-[2]-[1, 2, 4, 8].xlsx
[LogAnalysis] cache_file:results/results_cache-A6000-2-llama-13b-256-256-[2]-[1, 2, 4, 8].json
[LogAnalysis] exist cache_file:results/results_cache-A6000-2-llama-13b-256-256-[2]-[1, 2, 4, 8].json
[translate_to_excel_according_metrics] output_path:results/results_cache-A6000-2-llama-13b-256-256-[2]-[1, 2, 4, 8].xlsx
[LogAnalysis] cache_file:results/results_cache-A6000-2-llama-13b-512-256-[2]-[1, 2, 4, 8].json
[LogAnalysis] exist cache_file:results/results_cache-A6000-2-llama-13b-512-256-[2]-[1, 2, 4, 8].json
[translate_to_excel_according_metrics] output_path:results/results_cache-A6000-2-llama-13b-512-256-[2]-[1, 2, 4, 8].xlsx


/root/anaconda3/envs/llumnix/lib/python3.10/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")
/root/anaconda3/envs/llumnix/lib/python3.10/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")
/root/anaconda3/envs/llumnix/lib/python3.10/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


,request_time,prefill_time,decode_time,prefill_waiting_time,708a12a037924c318ac5f1b8aa716d63,9d3cea9934224af587c7988a0115b568,c8c7f98adb164de3b8dc2666d11db49f,eaa2f476acf244168fe297ce0fbe9f70,avg_speed,avg_migration_time,avg_migrate_waiting_time,avg_migration_count,avg_migration_aborted_dst_count,sum_migration_aborted_dst_count,reject_migrate_out_count,reject_migrate_in_count,3e32db4b986f4bebaf1f3e14b2e0e026,91d9d505831a4aa7908cf0a4ce69cecf,d5014c63cc00495f8cf5ef87fb5e7dfd,21a9850477f44f148445a0956d9ed82a,407944b7b59d4e3d94e80445f2a5a8fc,713c64e446ec4ab3ac65b495156efa7a,33f2414e7e8c4ec6a4715df754a5e65c,d569df3e8d0e4a37b136a3230b146f84
"labels_1,1,1,1",14.3529,257.1570,55.1125,175.0700,"{'decode_bs': 6.9804, 'decode_ratio': 0.9345, 'gpu_cache_usage': 0.191145, 'num_running_requests': 6.980083, 'num_waiting_requests': 0, 'num_killed_requests': 0, 'sm_active': 0.792235, 'mofc': 0.961169, 'prefill_step_time': 134.505859, 'decode_step_time': 49.192714}","{'decode_bs': 7.1324, 'decode_ratio': 0.9311, 'gpu_cache_usage': 0.195173, 'num_running_requests': 7.129099, 'num_waiting_requests': 0, 'num_killed_requests': 0, 'sm_active': 0.788901, 'mofc': 0.959787, 'prefill_step_time': 141.96197, 'decode_step_time': 49.37993}","{'decode_bs': 6.9706, 'decode_ratio': 0.9313, 'gpu_cache_usage': 0.190928, 'num_running_requests': 6.972959, 'num_waiting_requests': 0, 'num_killed_requests': 0, 'sm_active': 0.788791, 'mofc': 0.960213, 'prefill_step_time': 141.551587, 'decode_step_time': 49.175332}","{'decode_bs': 6.9373, 'decode_ratio': 0.9337, 'gpu_cache_usage': 0.189975, 'num_running_requests': 6.937825, 'num_waiting_requests': 0, 'num_killed_requests': 0, 'sm_active': 0.788993, 'mofc': 0.961428, 'prefill_step_time': 141.24912, 'decode_step_time': 49.095872}",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"labels_1,1-2",14.4322,206.4012,55.6303,164.8519,NaN,NaN,NaN,NaN,61.3642,620.3919,74.9354,1.04,0.04,80.0,8.0,80.0,"{'inference_type': 'decode', 'decode_bs': 26.4284, 'decode_all_time_bs': 26.4016, 'gpu_cache_usage': 0.226356, 'num_running_requests': 26.401576, 'num_waiting_requests': 0, 'num_killed_requests': 0, 'sm_active': 0.518323, 'mofc': 0.957549, 'decode_step_time': 46.013306}","{'inference_type': 'prefill', 'prefill_bs': 532.3059, 'prefill_all_time_bs': 120.3125, 'gpu_cache_usage': 0.027782, 'num_running_requests': 0.394545, 'num_waiting_requests': 0, 'num_killed_requests': 0, 'sm_active': 0.079392, 'mofc': 0.089547, 'prefill_step_time': 134.598376}","{'inference_type': 'prefill', 'prefill_bs': 531.9796, 'prefill_all_time_bs': 133.9229, 'gpu_cache_usage': 0.02978, 'num_running_requests': 0.436743, 'num_waiting_requests': 0, 'num_killed_requests': 0, 'sm_active': 0.124657, 'mofc': 0.083685, 'prefill_step_time': 134.751956}",NaN,NaN,NaN,NaN,NaN
"labels_1,1,1",15.6545,269.3581,60.1629,181.3185,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'decode_bs': 10.1647, 'decode_ratio': 0.9068, 'gpu_cache_usage': 0.279459, 'num_running_requests': 10.202446, 'num_waiting_requests': 0, 'num_killed_requests': 0, 'sm_active': 0.794872, 'mofc': 0.947173, 'prefill_step_time': 142.45953, 'decode_step_time': 51.831183}","{'decode_bs': 10.2387, 'decode_ratio': 0.9103, 'gpu_cache_usage': 0.28064, 'num_running_requests': 10.248718, 'num_waiting_requests': 0, 'num_killed_requests': 0, 'sm_active': 0.795566, 'mofc': 0.94811, 'prefill_step_time': 139.174923, 'decode_step_time': 51.67203}","{'decode_bs': 10.2157, 'decode_ratio': 0.9083, 'gpu_cache_usage': 0.280307, 'num_running_requests': 10.236817, 'num_waiting_requests': 0, 'num_killed_requests': 0, 'sm_active': 0.79758, 'mofc': 0.948167, 'prefill_step_time': 144.427304, 'decode_step_time': 51.869773}",NaN,NaN
labels_1-2,13.3246,250.0556,51.1035,183.5690,NaN,NaN,NaN,NaN,73.0983,484.0273,23.9854,1.00,0.00,0.0,42.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,"{'inference_type': 'decode', 'decode_bs': 24.5057, 'decode_all_time_bs': 24.4892, 'gpu_cache_usage': 0.208849, 'num_running_requests': 24.489203, 

In [6]:
model = 'llama-13b' # 'llama-7b'
instance_deploy_msg = [     # (prefill_tps, decode_tps)
    ([1,1,1,1],[]),([1,1],[2]),([1,1,1],[]),([1],[2]),
]
analysis = LogAnalysis_new(model,[2],[1,2,4,8],instance_deploy_msg,'128-256')
analysis.get_all_msg()  # 首次需要
metrics = [
        ["prefill_step_time", "decode_step_time", 'avg_migration_time','avg_migrate_waiting_time'],
        ["mofc", "decode_bs", "decode_ratio", "sum_migration_aborted_dst_count"],
        ["mofc", "prefill-mofc", "decode-mofc",]
]
analysis.translate_to_excel_according_metrics(metrics, suffix="more_info")

[LogAnalysis] cache_file:results/results_cache-A6000-2-llama-13b-128-256-[2]-[1, 2, 4, 8].json
[LogAnalysis] exist cache_file:results/results_cache-A6000-2-llama-13b-128-256-[2]-[1, 2, 4, 8].json
[translate_to_excel_according_metrics] output_path:results/results_cache-A6000-2-llama-13b-128-256-[2]-[1, 2, 4, 8]_more_info.xlsx


/root/anaconda3/envs/llumnix/lib/python3.10/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")
/root/anaconda3/envs/llumnix/lib/python3.10/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")
/root/anaconda3/envs/llumnix/lib/python3.10/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


In [7]:

model = 'llama-13b' # 'llama-7b'
instance_deploy_msg = [     # (prefill_tps, decode_tps)
    ([1,1,1,1],[]),([1,1],[2]),# ([1,1,1],[]),([1],[2]),
]
for req_len in ['2016-32']:
    analysis = LogAnalysis_new(model,[2],[1,2,4,8],instance_deploy_msg,req_len)
    analysis.get_all_msg()  # 首次需要
    # analysis.get_all_msg_updata_instance_metric()
    metrics = [
            ["request_time", "prefill_time", "decode_time"],
            'prefill_bs',
            'prefill_all_time_bs'
    ]
    analysis.translate_to_excel_according_metrics(metrics)  # 将结果转换为Excel格式

fix_fields = {
    'qps': str(2),
    # 'labels': "1,1,1,1"
    'concurrency': 4 # [1,2,4,8,16]
}
change_field = {
    'labels': ["1,1,1,1", "1,1-2"]
}

assert len(change_field) == 1

data = []
key = list(change_field.keys())[0]
values = list(change_field.values())[0]
for v in values:
    concurrency = get_value('concurrency', fix_fields, v)
    qps = get_value('qps', fix_fields, v)
    label = get_value('labels', fix_fields, v)
    data.append(analysis.results[concurrency][qps][label])
# 显示所有列
pd.set_option('display.max_columns', None)
# 显示所有行
pd.set_option('display.max_rows', None)
# 设置每列宽度（可选）
pd.set_option('display.max_colwidth', None)
df = pd.DataFrame(data, index=[f'{key}_{v}' for v in values])
df

[LogAnalysis] cache_file:results/results_cache-A6000-2-llama-13b-2016-32-[2]-[1, 2, 4, 8].json
[LogAnalysis] exist cache_file:results/results_cache-A6000-2-llama-13b-2016-32-[2]-[1, 2, 4, 8].json
[translate_to_excel_according_metrics] output_path:results/results_cache-A6000-2-llama-13b-2016-32-[2]-[1, 2, 4, 8].xlsx


/root/anaconda3/envs/llumnix/lib/python3.10/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


,request_time,prefill_time,decode_time,prefill_waiting_time,4e4cc0fb2dfc4b37b5413a663cf3b4ed,5b4777cfe6f34680bc5dd507170df2e5,60fa59d989d64146a63cb42eed6a5be4,b0e60ca46f3f45ca940cea330f72e5c2,avg_speed,avg_migration_time,avg_migrate_waiting_time,avg_migration_count,avg_migration_aborted_dst_count,sum_migration_aborted_dst_count,reject_migrate_out_count,reject_migrate_in_count,6ac022342723419f9493c44e5da54831,9b240a2b3b2646c69ad207b67d1c5451,e4f05c2cdc054bf1bbbd2fea62249545
"labels_1,1,1,1",2.5843,901.8612,53.8383,677.6970,"{'decode_bs': 1.6614, 'decode_ratio': 0.6243, 'gpu_cache_usage': 0.137068, 'num_running_requests': 1.591721, 'num_waiting_requests': 0.041757, 'num_killed_requests': 0, 'sm_active': 0.535449, 'mofc': 0.71617, 'prefill_step_time': 516.064113, 'decode_step_time': 45.961986}","{'decode_bs': 1.6537, 'decode_ratio': 0.627, 'gpu_cache_usage': 0.138066, 'num_running_requests': 1.603363, 'num_waiting_requests': 0.051699, 'num_killed_requests': 0, 'sm_active': 0.685473, 'mofc': 0.714503, 'prefill_step_time': 516.576204, 'decode_step_time': 46.353687}","{'decode_bs': 1.5881, 'decode_ratio': 0.6313, 'gpu_cache_usage': 0.132536, 'num_running_requests': 1.538936, 'num_waiting_requests': 0.032002, 'num_killed_requests': 0, 'sm_active': 0.619925, 'mofc': 0.719149, 'prefill_step_time': 521.611055, 'decode_step_time': 46.118686}","{'decode_bs': 1.6831, 'decode_ratio': 0.6262, 'gpu_cache_usage': 0.140124, 'num_running_requests': 1.627299, 'num_waiting_requests': 0.048979, 'num_killed_requests': 0, 'sm_active': 0.708773, 'mofc': 0.718349, 'prefill_step_time': 507.960718, 'decode_step_time': 46.452544}",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"labels_1,1-2",117.0902,103083.8892,451.3919,103031.5569,NaN,NaN,NaN,NaN,51.4917,2456.5854,10504.2908,3.482,2.482,4964.0,2943.0,4964.0,"{'inference_type': 'decode', 'decode_bs': 1.6222, 'decode_all_time_bs': 1.6194, 'gpu_cache_usage': 0.141357, 'num_running_requests': 1.619355, 'num_waiting_requests': 0, 'num_killed_requests': 0, 'sm_active': 0.595409, 'mofc': 0.294051, 'decode_step_time': 29.75426}","{'inference_type': 'prefill', 'prefill_bs': 2016, 'prefill_all_time_bs': 814.9951, 'gpu_cache_usage': 0.911525, 'num_running_requests': 8.706604, 'num_waiting_requests': 81.062238, 'num_killed_requests': 0, 'sm_active': 0.361651, 'mofc': 0.003134, 'prefill_step_time': 499.623495}","{'inference_type': 'prefill', 'prefill_bs': 2016, 'prefill_all_time_bs': 837.0342, 'gpu_cache_usage': 0.912289, 'num_running_requests': 8.693907, 'num_waiting_requests': 81.163202, 'num_killed_requests': 0, 'sm_active': 0.369468, 'mofc': 0.003552, 'prefill_step_time': 515.859382}"


In [8]:
model = 'llama-13b' # 'llama-7b'
instance_deploy_msg = [     # (prefill_tps, decode_tps)
    ([1,1,1,1],[]),([1,1],[2])# ,([1,1,1],[]),([1],[2]),
]
analysis = LogAnalysis_new(model,[2],[1,2,4,8],instance_deploy_msg,'2016-32')
analysis.get_all_msg()  # 首次需要
metrics = [
        # ["prefill_step_time", "decode_step_time", 'avg_migration_time','avg_migrate_waiting_time'],
        # ["prefill-mofc", "decode_bs", "decode_ratio", "sum_migration_aborted_dst_count"]
        ["prefill_step_time", "decode_step_time", 'avg_migration_time','avg_migrate_waiting_time'],
        ["mofc", "decode_bs", "decode_ratio", "sum_migration_aborted_dst_count"],
        ["mofc", "prefill-mofc", "decode-mofc",]
]
analysis.translate_to_excel_according_metrics(metrics, suffix="more_info")

[LogAnalysis] cache_file:results/results_cache-A6000-2-llama-13b-2016-32-[2]-[1, 2, 4, 8].json
[LogAnalysis] exist cache_file:results/results_cache-A6000-2-llama-13b-2016-32-[2]-[1, 2, 4, 8].json
[translate_to_excel_according_metrics] output_path:results/results_cache-A6000-2-llama-13b-2016-32-[2]-[1, 2, 4, 8]_more_info.xlsx


/root/anaconda3/envs/llumnix/lib/python3.10/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")
/root/anaconda3/envs/llumnix/lib/python3.10/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")
/root/anaconda3/envs/llumnix/lib/python3.10/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")
